# Notebook 02 — Normalización y revisión del tuning

Este notebook reproduce la normalización de las variables predictoras y de las coordenadas espaciales mediante escalamiento Min-Max. Además, documenta los resultados del tuning previamente ejecutado y compara su desempeño con los hiperparámetros por defecto usados como referencia metodológica.

El punto central del análisis es el **gap entre validación cruzada aleatoria y prueba espacial**. Aunque el grid search mejora el RMSE promedio en validación cruzada, sus hiperparámetros no superan el desempeño del modelo base en el grid de prueba 50 × 50. Esta diferencia sugiere que la validación cruzada aleatoria puede ser optimista cuando los datos tienen dependencia espacial.


## Compatibilidad con Colab y entorno local

La siguiente celda prepara el entorno de trabajo. En Google Colab instala las dependencias necesarias y clona el repositorio. En ejecución local, asume que el notebook se corre desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/linaq11/examen-2-gpboost.git"

if IN_COLAB:
    print("Entorno Colab detectado. Instalando dependencias...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gpboost", "shap"],
        check=True,
    )

    repo_dir = Path("/content/repo")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "-q", REPO_URL, str(repo_dir)],
            check=True,
        )

    os.chdir(repo_dir)
else:
    current_dir = Path.cwd()
    if current_dir.name == "notebooks":
        os.chdir(current_dir.parent)

print(f"Directorio de trabajo: {Path.cwd()}")


Directorio de trabajo: /sessions/optimistic-lucid-ptolemy/mnt/PERCEPCION REMOTA/Examen_2_GPBoost


In [2]:
import json
from itertools import product

import gpboost as gpb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
MODEL_DIR = Path("modelos")
FIGURE_DIR = Path("figuras")

for directory in (DATA_DIR, MODEL_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "simulado_train_seed1.csv"
TUNING_FILE = DATA_DIR / "tuning_resultados.json"
SCALER_FILE = MODEL_DIR / "scalers_minmax.json"

missing_files = [path for path in (TRAIN_FILE, TUNING_FILE) if not path.exists()]
if missing_files:
    missing = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(f"No se encontraron los archivos requeridos:\n{missing}")


/sessions/optimistic-lucid-ptolemy/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Datos y normalización Min-Max

Se cargan las variables simuladas, la variable respuesta y las coordenadas espaciales. Luego se ajusta un escalamiento Min-Max por separado para los predictores (`X1`, `X2`) y para las coordenadas (`s_1`, `s_2`). Los parámetros del escalamiento se guardan en `modelos/scalers_minmax.json` para que el preprocesamiento pueda replicarse posteriormente.


In [3]:
df = pd.read_csv(TRAIN_FILE)

feature_columns = ["X1", "X2"]
coord_columns = ["s_1", "s_2"]
target_column = "y"

X = df[feature_columns].to_numpy()
y = df[target_column].to_numpy()
coords = df[coord_columns].to_numpy()

feature_scaler = MinMaxScaler().fit(X)
coord_scaler = MinMaxScaler().fit(coords)

X_scaled = feature_scaler.transform(X)
coords_scaled = coord_scaler.transform(coords)

scaler_metadata = {
    "feature_columns": feature_columns,
    "coord_columns": coord_columns,
    "X_min": feature_scaler.data_min_.tolist(),
    "X_max": feature_scaler.data_max_.tolist(),
    "coords_min": coord_scaler.data_min_.tolist(),
    "coords_max": coord_scaler.data_max_.tolist(),
}

with SCALER_FILE.open("w", encoding="utf-8") as file:
    json.dump(scaler_metadata, file, indent=2, ensure_ascii=False)

print(f"Observaciones: {len(df)}")
print(f"Rango de X escalado: {X_scaled.min():.3f} – {X_scaled.max():.3f}")
print(f"Parámetros de escalamiento guardados en: {SCALER_FILE}")


Observaciones: 200
Rango de X escalado: 0.000 – 1.000
Parámetros de escalamiento guardados en: modelos/scalers_minmax.json


## 2. Espacio de búsqueda del tuning

El tuning evaluó combinaciones de `learning_rate`, `max_depth`, `min_data_in_leaf` y `num_boost_round`. La validación cruzada usada en esta etapa fue aleatoria con tres folds, por lo que sus resultados deben interpretarse con cautela en presencia de autocorrelación espacial.


In [4]:
param_grid = {
    "learning_rate": [0.005, 0.01, 0.05],
    "max_depth": [3, 5, 7],
    "min_data_in_leaf": [5, 10, 20],
    "num_boost_round": [247, 500],
}

param_combinations = list(product(*param_grid.values()))

print(f"Combinaciones evaluadas: {len(param_combinations)}")


Combinaciones evaluadas: 54


## 3. Resultados del tuning

El grid search ya fue ejecutado previamente. En esta celda se cargan sus resultados desde `data/tuning_resultados.json` para documentar la comparación sin repetir un proceso computacionalmente costoso.


In [5]:
with TUNING_FILE.open(encoding="utf-8") as file:
    tuning_results = json.load(file)

print(json.dumps(tuning_results, indent=2, ensure_ascii=False))


{
  "decision_final": "Modelo final = defaults del artículo (Notebook 01). Notebook 02 se conserva para documentar el gap CV-test como hallazgo metodológico.",
  "modelo_final": {
    "fuente": "Notebook 01 — defaults Sigrist 2021",
    "params": {
      "learning_rate": 0.01,
      "max_depth": 3,
      "min_data_in_leaf": 10,
      "num_leaves": 1024,
      "num_boost_round": 247
    },
    "RMSE_test": 0.6073,
    "archivo": "modelos/gpboost_basico.json"
  },
  "tuning_documentado_pero_descartado": {
    "razon": "Tres grids (8, 12 y 36 combinaciones) eligen consistentemente params (lr=0.10) que mejoran CV pero empeoran test. Es evidencia del gap CV-test por autocorrelación espacial, no error del procedimiento.",
    "grids_probados": [
      {
        "combinaciones": 8,
        "nfold": 3,
        "best_params": {
          "lr": 0.1,
          "depth": 10,
          "mindata": 10
        },
        "RMSE_test": 0.6816
      },
      {
        "combinaciones": 12,
        "nfold":

## 4. Interpretación: gap entre CV aleatoria y prueba espacial

El archivo de resultados resume tres grids progresivamente más exhaustivos. En los tres casos, la combinación con mejor RMSE de validación cruzada obtiene un peor desempeño en el grid de prueba 50 × 50 que los hiperparámetros por defecto del artículo.

| Grid | Mejor RMSE en CV | RMSE en test con esos parámetros | RMSE en test con defaults |
|---|---:|---:|---:|
| 8 combinaciones | 0.32 | 0.6816 | **0.6073** |
| 12 combinaciones | 0.36 | 0.6706 | **0.6073** |
| 36 combinaciones | 0.43 | 0.6619 | **0.6073** |

La diferencia entre CV y test es de aproximadamente 0.30 unidades de RMSE. Esto indica que los folds aleatorios comparten estructura espacial con el entrenamiento y pueden favorecer hiperparámetros que no generalizan bien fuera del patrón espacial muestreado. Por esta razón, se conservan los hiperparámetros por defecto de Sigrist (2021) como referencia para el modelo final.


## 5. Entrenamiento del modelo base

Se reentrena el modelo GPBoost con los hiperparámetros por defecto usados como referencia. Esta decisión prioriza la generalización espacial observada en el grid de prueba sobre el desempeño aparente de la validación cruzada aleatoria.


In [6]:
gp_model = gpb.GPModel(
    gp_coords=coords,
    cov_function="exponential",
)

default_params = {
    "objective": "regression_l2",
    "learning_rate": 0.01,
    "max_depth": 3,
    "min_data_in_leaf": 10,
    "num_leaves": 1024,
    "verbose": -1,
}

train_data = gpb.Dataset(X, y)

default_model = gpb.train(
    params=default_params,
    train_set=train_data,
    gp_model=gp_model,
    num_boost_round=247,
)

print("Modelo base reentrenado con los hiperparámetros por defecto.")


Modelo base reentrenado con los hiperparámetros por defecto.


## 6. Cierre metodológico

La principal conclusión de este notebook es que la validación cruzada aleatoria puede ser engañosa en datos con dependencia espacial. En este caso, el tuning mejora la métrica interna de CV, pero no mejora la generalización sobre el grid espacial de prueba. Por ello, la evaluación posterior debe apoyarse en estrategias de validación espacial, como la validación por bloques desarrollada en el Notebook 04.
